# Анализ графа вызовов и поиск сильно связных компонент (SCC)

Этот блокнот посвящен расширенным возможностям анализа кода в UnifyWeaver:

- **Построение графа вызовов** — построение графов зависимостей на основе кода Prolog
- **Поиск SCC** — обнаружение сильно связных компонент (взаимная рекурсия)
- **Анализ паттернов** — определение типов рекурсии
- **Визуализация зависимостей** — наглядное отображение связей между предикатами

## Цели обучения

- Понять, как UnifyWeaver анализирует структуру исходного кода
- Строить и исследовать графы вызовов
- Обнаруживать взаимную рекурсию с помощью алгоритма Тарьяна (Tarjan)
- Визуализировать зависимости в кодовой базе

## Настройка

Загрузка UnifyWeaver и модулей анализа.

In [ ]:
% Загрузить инициализацию
['../init'].

% Загрузить модули анализа
use_module(unifyweaver(core/advanced/call_graph)).
use_module(unifyweaver(core/advanced/scc_detection)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Пример 1: Простой граф вызовов

Начнем с простого предиката и построим его граф вызовов.

In [ ]:
% Определить предикат ancestor
:- dynamic ancestor/2.
:- dynamic parent/2.

% Факты parent
parent(abraham, isaac).
parent(isaac, jacob).

% Правила ancestor
ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

### Построение графа вызовов

In [ ]:
% Построить граф вызовов для ancestor
build_call_graph([ancestor/2], _Graph),
format('Call Graph for ancestor/2:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Анализ зависимостей

In [ ]:
% Получить все зависимости ancestor/2
get_dependencies(ancestor/2, _Deps),
format('Dependencies of ancestor/2: ~w~n', [_Deps]).

% Проверить, является ли саморекурсивным
(is_self_recursive(ancestor/2) ->
    writeln('✓ ancestor/2 is self-recursive')
;
    writeln('✗ ancestor/2 is not self-recursive')
).

## Пример 2: Обнаружение взаимной рекурсии

Теперь определим взаимную рекурсию на примере четных и нечетных чисел.

In [ ]:
% Определить взаимно рекурсивные предикаты
:- dynamic is_even/1.
:- dynamic is_odd/1.

is_even(0).
is_even(N) :- N > 0, N1 is N - 1, is_odd(N1).

is_odd(1).
is_odd(N) :- N > 1, N1 is N - 1, is_even(N1).

### Построение графа вызовов для обоих предикатов

In [ ]:
% Построить граф вызовов для обоих предикатов
build_call_graph([is_even/1, is_odd/1], _Graph),
format('Call Graph for is_even/1 and is_odd/1:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Поиск сильно связных компонент (SCC)

In [ ]:
% Перестроить граф, поскольку переменные не сохраняются между ячейками блокнота
build_call_graph([is_even/1, is_odd/1], _Graph),
% Найти SCC с помощью алгоритма Тарьяна
find_sccs(_Graph, _SCCs),
format('Strongly Connected Components:~n'),
forall(member(_SCC, _SCCs),
    format('  ~w~n', [_SCC])).

### Проверка тривиальности SCC

In [ ]:
% Пересчитать производные значения, чтобы эта ячейка также выполнялась независимо
build_call_graph([is_even/1, is_odd/1], _Graph),
find_sccs(_Graph, _SCCs),
% Проверить каждую SCC
forall(member(_SCC, _SCCs),
    (   is_trivial_scc(_SCC) ->
        format('  ~w is trivial (single predicate)~n', [_SCC])
    ;
        format('  ~w is NON-TRIVIAL (mutual recursion!)~n', [_SCC])
    )
).

## Пример 3: Сложный граф вызовов

Проанализируем более сложную систему с несколькими предикатами.

In [ ]:
% Определить небольшую программу с несколькими предикатами
:- dynamic grandparent/2.
:- dynamic sibling/2.
:- dynamic cousin/2.

% grandparent использует parent
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).

% sibling: одинаковые родители, разные дети
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% cousin: родители — братья или сестры
cousin(X, Y) :- parent(P1, X), parent(P2, Y), sibling(P1, P2).

### Построение полного графа вызовов

In [ ]:
% Построить граф вызовов для всех предикатов
build_call_graph([grandparent/2, sibling/2, cousin/2], _Graph),
format('Complete Call Graph:~n'),
forall(member(_Edge, _Graph),
    format('  ~w~n', [_Edge])).

### Поиск групп предикатов

Поиск группы взаимно рекурсивных предикатов, содержащей начальный предикат.

In [ ]:
% Найти группу взаимной рекурсии, содержащую cousin/2
predicates_in_group(cousin/2, _Group),
format('Mutually recursive group containing cousin/2: ~w~n', [_Group]).

## Пример 4: Распознавание паттернов

Используем модули сопоставления с паттернами для классификации типов рекурсии.

In [ ]:
% Определить различные паттерны рекурсии
:- dynamic count/3.     % Хвостовая рекурсия
:- dynamic factorial/2. % Линейная рекурсия
:- dynamic fib/2.       % Древовидная рекурсия (или линейная, если обнаружена)

% Хвосторекурсивный подсчет
count([], Acc, Acc).
count([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count(T, Acc1, N).

% Линейно-рекурсивный факториал
factorial(0, 1).
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),
    F is N * F1.

% Фибоначчи (может определяться как линейная или древовидная рекурсия)
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.

### Распознавание хвостовой рекурсии

In [ ]:
% Проверить, является ли count/3 хвосторекурсивным
(is_tail_recursive_accumulator(count/3, _AccInfo) ->
    format('✓ count/3 is tail recursive: ~w~n', [_AccInfo])
;
    writeln('✗ count/3 is not tail recursive')
).

### Распознавание линейной рекурсии

In [ ]:
% Проверить, является ли factorial/2 линейно-рекурсивным
(is_linear_recursive_streamable(factorial/2) ->
    writeln('✓ factorial/2 is linear recursive')
;
    writeln('✗ factorial/2 is not linear recursive')
).

### Подсчет рекурсивных вызовов

In [ ]:
% Подсчитать рекурсивные вызовы в fibonacci
functor(_FibHead, fib, 2),
user:clause(_FibHead, _FibBody),
once(contains_call_to(_FibBody, fib)),
count_recursive_calls(_FibBody, fib, _Count),
format('Fibonacci body has ~w recursive calls~n', [_Count]).

## Визуализация в формате DOT

Сгенерируем представление графа вызовов на языке Graphviz DOT.

In [ ]:
% Вспомогательная функция для генерации формата DOT
generate_dot(Graph, DotCode) :-
    findall(Line,
        (   member(From -> To, Graph),
            format(atom(Line), '  "~w" -> "~w";', [From, To])
        ),
        Lines),
    atomic_list_concat(['digraph CallGraph {', '  rankdir=LR;' | Lines], '\n', Body),
    format(atom(DotCode), '~w~n}~n', [Body]).

% Сгенерировать DOT для графа четных/нечетных
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
writeln('DOT Code for even/odd call graph:'),
writeln(_DotCode).

### Сохранение файла DOT

In [ ]:
% Перестроить исходный код DOT, так как переменные не сохраняются между ячейками
build_call_graph([is_even/1, is_odd/1], _Graph),
generate_dot(_Graph, _DotCode),
setup_call_cleanup(
    open('/shared/data/even_odd_graph.dot', write, _Stream),
    write(_Stream, _DotCode),
    close(_Stream)),
writeln('✓ Saved to /shared/data/even_odd_graph.dot').

writeln('To visualize, run:'),
writeln('  dot -Tpng /shared/data/even_odd_graph.dot -o even_odd_graph.png').

## Упражнение: Анализ собственного кода

Попробуйте определить свои собственные предикаты и проанализировать их структуру!

In [ ]:
% Определите свои предикаты здесь
% Затем постройте графы вызовов, найдите SCC и определите паттерны

% Пример:
% :- dynamic my_predicate/2.
% my_predicate(...) :- ...

% build_call_graph([my_predicate/2], Graph).


## Резюме

В этом блокноте вы изучили:

✅ Как строить графы вызовов из кода Prolog

✅ Как находить сильно связные компоненты (SCC) для взаимной рекурсии

✅ Как использовать сопоставители паттернов для классификации типов рекурсии

✅ Как анализировать зависимости между предикатами

✅ Как визуализировать графы вызовов с помощью формата DOT

## Дополнительные темы

Для более глубокого анализа:

- **Топологическая сортировка**: используйте `topological_order/2` для упорядочивания SCC по зависимостям
- **Пользовательские сопоставители паттернов**: создавайте собственные предикаты для анализа паттернов
- **Извлечение паттерна аккумулятора**: используйте `extract_accumulator_pattern/2` для детального анализа
- **Запрет линейной рекурсии**: используйте `forbid_linear_recursion/1` для принудительного выбора других стратегий компиляции

## Связанные файлы и источники

- Глава 10: Интроспекция и теория Prolog
- `src/unifyweaver/core/advanced/call_graph.pl`
- `src/unifyweaver/core/advanced/scc_detection.pl`
- `src/unifyweaver/core/advanced/pattern_matchers.pl`